## Анализ вероятности ухода клиента из банка

In [10]:
import numpy as np
import pandas as pd
import joblib
import time
import warnings

from itertools import product
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


warnings.filterwarnings("ignore", message="X does not have valid feature names*")
RANDOM_STATE = 42

In [2]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [3]:
X, y = train.drop(columns=['id', 'Churn']), train['Churn'].map({'No': 0, 'Yes': 1})

In [4]:
numeric_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

X = preprocessor.fit_transform(X)
y = y.to_numpy()

### Создание Baseline

In [6]:
class modelsHub:
    def __init__(self):
        self.results = []
        self.models = {}

    def _get_scores(self, model, X):
        if hasattr(model, 'predict_proba'):
            return model.predict_proba(X)[:, 1]
        if hasattr(model, 'decision_function'):
            return model.decision_function(X)
        return None

    def choice_best(self, model_class, X_train, y_train, model_name, params, general_params=None, n_splits=5):
        best_params = None
        best_auc = -np.inf
        
        for i in tqdm(range(len(params)), total=len(params), desc=f'{model_name} tuning'):
            params_to_use = {**params[i], **general_params} if general_params else params[i]
            try:
                row = self.fit_predict(
                    model_class,
                    X_train,
                    y_train,
                    model_name=model_name,
                    params=params_to_use,
                    n_splits=n_splits,
                    return_model=False
                )

                if row['roc_auc'] > best_auc:
                    best_params = params_to_use
                    best_auc = row['roc_auc']
            except Exception as e:
                print(f"Error with params {params_to_use}: {e}")

        # обучаем лучшую модель на всём train
        best_model = model_class(**best_params)
        best_model.fit(X_train, y_train)
        self.models[model_name] = best_model

        return best_model, best_params

    def fit_predict(self, model_class, X, y, model_name, params=None, n_splits=5, return_model=True):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

        metrics = {
            'accuracy': [],
            'precision': [],
            'recall': [],
            'f1': [],
            'roc_auc': []
        }
        fit_times = []
        pred_times = []

        for train_idx, valid_idx in skf.split(X, y):
            X_train_fold, X_valid_fold = X[train_idx], X[valid_idx]
            y_train_fold, y_valid_fold = y[train_idx], y[valid_idx]

            model = model_class(**params) if params else model_class()

            fit_start = time.perf_counter()
            model.fit(X_train_fold, y_train_fold)
            fit_times.append(time.perf_counter() - fit_start)

            pred_start = time.perf_counter()
            y_pred = model.predict(X_valid_fold)
            pred_times.append(time.perf_counter() - pred_start)

            y_score = self._get_scores(model, X_valid_fold)
            metrics['accuracy'].append(accuracy_score(y_valid_fold, y_pred))
            metrics['precision'].append(precision_score(y_valid_fold, y_pred, zero_division=0))
            metrics['recall'].append(recall_score(y_valid_fold, y_pred, zero_division=0))
            metrics['f1'].append(f1_score(y_valid_fold, y_pred, zero_division=0))
            metrics['roc_auc'].append(roc_auc_score(y_valid_fold, y_score))

        row = {
            'model': model_name,
            'fit_time_sec': np.mean(fit_times),
            'predict_time_sec': np.mean(pred_times),
            'accuracy': np.mean(metrics['accuracy']),
            'precision': np.mean(metrics['precision']),
            'recall': np.mean(metrics['recall']),
            'f1': np.mean(metrics['f1']),
            'roc_auc': np.nanmean(metrics['roc_auc']),
            'params': params
        }
        self.results.append(row)

        if return_model:
            final_model = model_class(**params) if params else model_class()
            final_model.fit(X, y)
            return row, final_model

        return row

    def leaderboard(self, only_best=False):
        if not self.results:
            return pd.DataFrame()

        df = pd.DataFrame(self.results)

        if only_best:
            def _best_index(g):
                if g['roc_auc'].notna().any():
                    return g['roc_auc'].idxmax()
                return g.index[0]
            best_indices = df.groupby('model', group_keys=False).apply(_best_index).values
            df = df.loc[best_indices].reset_index(drop=True)

        df_sorted = df.sort_values('roc_auc', ascending=False).reset_index(drop=True)
        df['params_str'] = df['params'].apply(lambda x: str(x))

        def highlight_best_roc_auc(row):
            best_roc_auc = df_sorted['roc_auc'].max()
            if pd.notna(row['roc_auc']) and row['roc_auc'] == best_roc_auc:
                return ['background-color: #F0FFF0'] * len(row)
            return [''] * len(row)

        return (
            df_sorted.style
            .format({
                'fit_time_sec': '{:.4f}',
                'predict_time_sec': '{:.4f}',
                'accuracy': '{:.4f}',
                'precision': '{:.4f}',
                'recall': '{:.4f}',
                'f1': '{:.4f}',
                'roc_auc': '{:.4f}',
            })
            .apply(highlight_best_roc_auc, axis=1)
        )

In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

hub = modelsHub()
hub.fit_predict(
    LogisticRegression,
    X_train,
    y_train,
    model_name='LogisticRegression',
)
hub.leaderboard()

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,LogisticRegression,0.5951,0.0024,0.8545,0.6848,0.6557,0.6699,0.9078,None


### RandomForest

In [ ]:
n_estimators = [100, 300]
max_depth = [None, 10, 20]
min_samples_leaf = [1, 2]
max_features = ['sqrt', 'log2']

param_grid = list(product(n_estimators, max_depth, min_samples_leaf, max_features))

random_forest, best_params = hub.choice_best(
    model_class=RandomForestClassifier,
    X_train=X_train,
    y_train=y_train,
    model_name='RandomForestClassifier',
    params=[
        {
            'n_estimators': n,
            'max_depth': d,
            'min_samples_leaf': l,
            'max_features': f
        }
        for n, d, l, f in param_grid
    ],
    general_params={
        'random_state': RANDOM_STATE,
        'n_jobs': -1
    },
    n_splits=3
)

random_forest = joblib.load("./saved models/random_forest.joblib")
print(best_params)

RandomForestClassifier(max_depth=10, min_samples_leaf=2, n_estimators=300,
                       n_jobs=-1, random_state=42)


### CatBoost

In [ ]:
depth = [6, 8, 10]
iterations = [100, 600, 5000]
reg_lambdas = [0.1, 0.5]

param_grid = list(product(depth, iterations, reg_lambdas))
catboost, best_params = hub.choice_best(
    model_class=CatBoostClassifier,
    X_train=X_train,
    y_train=y_train,
    model_name='CatBoostClassifier',
    params=[{'depth': d, 'iterations': it, 'reg_lambda': rl} for d, it, rl in param_grid],
    general_params={'random_state': RANDOM_STATE, 'verbose': 0, 'early_stopping_rounds': 50},
    eval=(X_valid, y_valid)
)

catboost.save_model("./saved models/catboost_model.cbm")

In [ ]:
hub.leaderboard(only_best=True)

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,CatBoostClassifier,95.3049,0.1411,0.8613,0.7130,0.6425,0.6759,0.9157,"{'depth': 6, 'iterations': 5000, 'reg_lambda': 0.5, 'random_state': 42, 'verbose': 0, 'early_stopping_rounds': 50}"
1,RandomForestClassifier,11.7599,0.4754,0.8567,0.7016,0.6329,0.6655,0.9115,"{'n_estimators': 300, 'max_depth': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'random_state': 42, 'n_jobs': -1}"
2,LogisticRegression,0.6574,0.0021,0.8545,0.6848,0.6557,0.6699,0.9078,None


### LightGBM

### Формирование ответа для kaggle

In [ ]:
model = ...

X_test = preprocessor.transform(test.drop(columns=['id']))

test_proba = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    'id': test['id'],
    'Churn': test_proba
})

submission.to_csv('submission.csv', index=False)
submission.head()

In [ ]:
num_leaves = [31, 63]
n_estimators = [100, 500]
learning_rates = [0.05, 0.1]
max_depths = [6, 10, -1]
reg_alphas = [0, 1]
reg_lambdas = [0, 1]

param_grid = list(product(num_leaves, n_estimators, learning_rates, max_depths, reg_alphas, reg_lambdas))
params = [
    {
        'num_leaves': nl,
        'n_estimators': ne,
        'learning_rate': lr,
        'max_depth': md,
        'reg_alpha': ra,
        'reg_lambda': rl
    }
    for nl, ne, lr, md, ra, rl in param_grid
]

lgbm_model, best_params = hub.choice_best(
    model_class=LGBMClassifier,
    X_train=X_train,
    y_train=y_train,
    model_name='LightGBM',
    params=params,
    general_params={'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbosity': -1},
    n_splits=3
)

joblib.dump(lgbm_model, "./saved models/lightgbm.joblib")
hub.leaderboard(only_best=True)

,model,fit_time_sec,predict_time_sec,accuracy,precision,recall,f1,roc_auc,params
0,LightGBM,2.5832,0.5105,0.8613,0.7119,0.6455,0.6771,0.9157,"{'num_leaves': 31, 'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 6, 'reg_alpha': 1, 'reg_lambda': 1, 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}"
1,LogisticRegression,0.5951,0.0024,0.8545,0.6848,0.6557,0.6699,0.9078,None
